In [2]:
# ライブラリのインポート
import pandas as pd #テキスト参照
from google.colab import drive #Google Colab専用のライブラリから、ドライブ操作用の機能（drive）を呼び出す
drive.mount('/content/drive') # GoogleDriveをマウント

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#課題 1: 複数の日付のデータを一つのデータフレームにまとめる

In [3]:
from pathlib import Path


def resolve_data_storage() -> Path:
    """データ集計・DataStorage フォルダを特定（Colab / ローカル）。"""
    candidates: list[Path] = []

    # スクリプトと同じディレクトリ直下（.py 実行時）
    try:
        here = Path(__file__).resolve().parent
        candidates.append(here / "データ集計・DataStorage")
    except NameError:
        pass  # ノートブック等で __file__ がない場合

    # Google Colab（Drive マウント後）
    drive_my = Path("/content/drive/MyDrive")
    if drive_my.is_dir():
        candidates.append(drive_my / "OMNIA/考える/アナリティクス/データ集計・DataStorage")
        candidates.append(drive_my / "OMNIA/アナリティクス/データ集計・DataStorage")

    # カレントディレクトリ基準
    candidates.append(Path("データ集計・DataStorage"))
    candidates.append(Path.cwd() / "データ集計・DataStorage")

    for path in candidates:
        if path.is_dir():
            return path.resolve()

    tried = "\n".join(f"  - {p}" for p in candidates)
    raise FileNotFoundError(f"データ集計・DataStorage が見つかりません。試したパス:\n{tried}")


DATA_STORAGE = resolve_data_storage()

# CSVファイルのパスを指定
order_dir = DATA_STORAGE / "order_data"
paths_orders = sorted(order_dir.glob("order_data_2022_12_*.csv"))

# 各CSVファイルを読み込み、リストに格納
dfs_orders = [pd.read_csv(p) for p in paths_orders]

# データフレームを統合
df_orders = pd.concat(dfs_orders, ignore_index=True)

# 統合したデータフレームの先頭5行を表示
df_orders.head()

,order_no,time_type,order_state,receive_type,approve_date,created_at,driver_id,delivery_date,latest_delivery_date,accept_date,pickup_date,pass_date,local_area_flag
0,BY534M,1,4,2,2022/12/1 0:00,2022/12/1 0:00,5606.0,2022/12/1 0:53,2022/12/1 1:23,2022/12/1 0:07,2022/12/1 0:23,2022/12/1 0:44,0
1,GV384O,1,4,2,2022/12/1 0:12,2022/12/1 0:03,76304.0,2022/12/1 0:36,2022/12/1 1:06,2022/12/1 0:14,2022/12/1 0:26,2022/12/1 0:48,0
2,WK460E,1,4,2,2022/12/1 0:06,2022/12/1 0:05,17544.0,2022/12/1 0:48,2022/12/1 1:18,2022/12/1 0:13,2022/12/1 0:17,2022/12/1 0:32,0
3,LH063F,1,4,2,2022/12/1 0:06,2022/12/1 0:06,15018.0,2022/12/1 0:49,2022/12/1 1:19,2022/12/1 0:10,2022/12/1 0:16,2022/12/1 0:45,0
4,DK120M,1,4,2,2022/12/1 0:09,2022/12/1 0:07,26444.0,2022/12/1 0:35,2022/12/1 1:05,2022/12/1 0:11,2022/12/1 0:38,2022/12/1 0:43,0


# 課題 2: データの重複行を削除

In [ ]:
# 課題1の df_orders を利用（上のセルを実行済みであること）
df_dedup = df_orders.copy()
n_before = len(df_dedup)
n_dup = df_dedup.duplicated().sum()
df_dedup = df_dedup.drop_duplicates().reset_index(drop=True)
n_after = len(df_dedup)
print(f"削除前: {n_before} 行, 重複: {n_dup} 行, 削除後: {n_after} 行")
df_dedup.head()

※データフレームの行数を求める

In [ ]:
paths_orders = sorted((DATA_STORAGE / "order_data").glob("order_data_2022_12_*.csv"))
dfs_orders = [pd.read_csv(p) for p in paths_orders]
df_orders_rowcount = pd.concat(dfs_orders, ignore_index=True)
print("行数:", len(df_orders_rowcount))

# 課題 3: データをマージする

In [ ]:
orders_m = df_orders.copy()

paths_deliv = sorted(
    (DATA_STORAGE / "delivery_status_history_data").glob("delivery_status_history_data_2022_12_*.csv")
)
df_delivery = pd.concat([pd.read_csv(p) for p in paths_deliv], ignore_index=True)

paths_drv = sorted(
    (DATA_STORAGE / "driver_shop_arrival_history_data").glob(
        "driver_shop_arrival_history_data_2022_12_*.csv"
    )
)
df_driver = pd.concat([pd.read_csv(p) for p in paths_drv], ignore_index=True)

merged = orders_m.merge(df_delivery, on="order_no", how="left")
merged = merged.merge(df_driver, on="order_no", how="left")
merged.head()

# 課題 4: データをフィルターして抽出する


In [ ]:
# 課題3の merged を使用（注文完了かつデリバリーの例）
filtered = merged[(merged["order_state"] == 4) & (merged["receive_type"] == 2)].copy()
print(len(merged), "→", len(filtered))
filtered.head()

# 課題 5: データを整形する

In [ ]:
def unix_to_datetime(series):
    return pd.to_datetime(series, unit="s", errors="coerce")


df_shape = filtered.copy()
if "created_date" in df_shape.columns:
    df_shape["created_date"] = unix_to_datetime(df_shape["created_date"])
if "created_at_x" in df_shape.columns:
    df_shape = df_shape.rename(columns={"created_at_x": "created_at"})
df_shape.head()

# 課題 6: for文を使ってループ処理をする

In [ ]:
date_cols = [
    "approve_date",
    "created_at",
    "delivery_date",
    "latest_delivery_date",
    "accept_date",
    "pickup_date",
    "pass_date",
    "shop_arrival_at",
    "created_at_y",
]
df_loop = df_shape.copy()
for col in date_cols:
    if col in df_loop.columns:
        df_loop[col] = pd.to_datetime(df_loop[col], errors="coerce")
df_loop.dtypes

# 課題 9: スプレッドシートへのデータ抽出

In [ ]:
export_path = DATA_STORAGE / "課題2_export_for_spreadsheet.csv"
df_loop.to_csv(export_path, index=False, encoding="utf-8-sig")
print("出力:", export_path)

## 追加: `データ集計・DataStorage` を使った簡易分析

- **前提**: 課題1（`df_orders` / `df_dedup`）・課題3（`merged`）・課題6（`df_loop`）まで実行済みであること。
- **内容**: 日次注文数、`order_state` / `receive_type` の分布、マージ後の結合率、店舗到着〜受付までのリードタイム（分）の記述統計、参考として `join_orders`（2021/9）の日次件数。

In [ ]:
# --- 分析: order_data（4日分・重複除去後）とマージ結果 ---

# 【1】承認日別 注文数
od = df_dedup.copy()
od["approve_day"] = pd.to_datetime(od["approve_date"], errors="coerce").dt.date
print("【1】承認日別 注文数")
print(od.groupby("approve_day", dropna=False).size().to_string())
print()

# 【2】【3】注文状態・受取タイプ
print("【2】order_state 別件数")
print(od["order_state"].value_counts(dropna=False).sort_index().to_string())
print()
print("【3】receive_type 別件数（教材の定義に準拠）")
print(od["receive_type"].value_counts(dropna=False).sort_index().to_string())
print()

# 【4】マージ後の結合率（補助テーブルが付いた割合）
n_m = len(merged)
nz_del = merged["created_date"].notna().sum() if "created_date" in merged.columns else 0
nz_arr = merged["shop_arrival_at"].notna().sum() if "shop_arrival_at" in merged.columns else 0
print("【4】マージ後: 補助データの付与率")
print(f"注文行数: {n_m}")
if n_m:
    print(f"  delivery_status_history あり: {nz_del} ({100 * nz_del / n_m:.1f}%)")
    print(f"  driver_shop_arrival あり: {nz_arr} ({100 * nz_arr / n_m:.1f}%)")
print()

# 【5】店舗到着 〜 受付(accept) までのリードタイム（分）
if "shop_arrival_at" in df_loop.columns and "accept_date" in df_loop.columns:
    tmp = df_loop.dropna(subset=["shop_arrival_at", "accept_date"]).copy()
    tmp["lead_min"] = (tmp["accept_date"] - tmp["shop_arrival_at"]).dt.total_seconds() / 60.0
    print("【5】店舗到着〜受付までの分数（分）※負の値はデータ不整合の可能性")
    print(tmp["lead_min"].describe(percentiles=[0.5, 0.9, 0.99]).to_string())
else:
    print("【5】スキップ: shop_arrival_at / accept_date がありません（課題6まで実行してください）")
print()

# 【参考】join_orders（2021/9/25-28）日次件数
jo_dir = DATA_STORAGE / "join_orders"
jo_paths = sorted(jo_dir.glob("join_orders_2021_09_*.csv"))
if jo_paths:
    jdf = pd.concat([pd.read_csv(p, sep="\t") for p in jo_paths], ignore_index=True)
    jdf["order_dt"] = pd.to_datetime(jdf["order_date"], errors="coerce")
    jdf["d"] = jdf["order_dt"].dt.date
    print("【参考】join_orders 日次件数（2021/9/25〜28 ファイル群）")
    print(jdf.groupby("d", dropna=False).size().to_string())
else:
    print("【参考】join_orders の CSV が見つかりませんでした")